In [ ]:
%load_ext autoreload
%autoreload 2

# Spectrum viewer

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from Code.Utils.util_methods import UtilMethods
from sklearn.model_selection import train_test_split
import numpy as np
#from jupyter_datatables import init_datatables_mode

base = UtilMethods.find_project_root(os.getcwd())
print(f"Project root found: {base}")

if load_dotenv(f'{base}/.env'):
    print(".env found")
else:
    print("ERROR .env not found")

## Provide the data (.csv or .pkl)

In [ ]:
PATH = f'{base}/Dataset/unique_recipes.pkl'
ROWS = [1655, 3532]

## DON'T TOUCH FROM HERE!!

In [ ]:
spectrum_columns = [f'{i}nm' for i in range(400, 741, 10)]
#spectrum_columns

In [ ]:
if PATH.split('.')[-1] == 'pkl':
    spectra_df = pd.read_pickle(PATH).loc[ROWS, spectrum_columns]
elif PATH.split('.')[-1] == 'csv':
    spectra_df = pd.read_csv(PATH).loc[ROWS, spectrum_columns]

#testing:

# create a row with all zeros
new_row = dict.fromkeys(spectra_df.columns, 0)

# set one column to your value
new_row["470nm"] = 77
new_row["570nm"] = 58

# add it as a new row
spectra_df.loc[len(spectra_df)] = new_row

spectra_df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import colour  # colour-science

# helper: map wavelength in nm to srgb using cie 1931 2° cmfs and d65 white
def wl_to_srgb(wavelength_nm):
    # wavelength_to_XYZ accepts scalars or arrays and returns xyz tristimulus
    xyz = colour.wavelength_to_XYZ(wavelength_nm)
    # convert to srgb under d65
    rgb = colour.XYZ_to_sRGB(xyz)
    # values can be slightly outside gamut, clip to displayable range
    return np.clip(rgb, 0.0, 1.0)

# helper: build a smooth spectrum background
def build_spectrum(wl_min=380, wl_max=780, n=600):
    wls = np.linspace(wl_min, wl_max, n)
    rgbs = wl_to_srgb(wls)  # shape (n, 3)
    return wls, rgbs

# precompute spectrum background colors
bg_wls, bg_cols = build_spectrum(380, 780, 700)

# loop over each row with individual plots in corresponding color
for idx, row in spectra_df.iterrows():
    # convert column names like "400nm" -> 400 as int
    x = spectra_df.columns.str.replace("nm", "", regex=False).astype(int).to_numpy()
    y = row.to_numpy()

    fig, ax = plt.subplots(figsize=(12, 4), dpi=100)

    # draw spectrum background
    for i in range(len(bg_wls) - 1):
        ax.axvspan(bg_wls[i], bg_wls[i + 1], color=bg_cols[i], alpha=0.3)

    # compute reflectance weighted centroid wavelength to color the curve
    centroid_nm = np.sum(x * y) / np.sum(y)
    curve_color = wl_to_srgb(centroid_nm)

    # plot reflectance curve in its corresponding color
    ax.scatter(x, y, color=curve_color)
    ax.plot(x, y, linestyle='-', marker='o', color=curve_color)

    # axes and layout
    ax.set_xlabel("wavelength (nm)")
    ax.set_ylabel("reflectance (%)")
    ax.set_title(f"row {idx}")
    ax.set_ylim(0, 100)
    ax.set_xlim(390, 750)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

# combined plot with all curves colored by their centroid color
fig, ax = plt.subplots(figsize=(12, 4), dpi=100)

# reuse x for all rows
x = spectra_df.columns.str.replace("nm", "", regex=False).astype(int).to_numpy()

# spectrum background
for i in range(len(bg_wls) - 1):
    ax.axvspan(bg_wls[i], bg_wls[i + 1], color=bg_cols[i], alpha=0.3)

# plot each curve with its cie-based color
for idx, row in spectra_df.iterrows():
    y = row.to_numpy()
    centroid_nm = np.sum(x * y) / np.sum(y)
    curve_color = wl_to_srgb(centroid_nm)
    ax.plot(x, y, marker='o', color=curve_color, label=f"row {idx}")

ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("reflectance (%)")
ax.set_title("all curves with cie-based colors")
ax.set_ylim(0, 100)
ax.set_xlim(380, 780)
ax.legend() 
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()
